# Logit Standardization KD — run all experiments on a free GPU (Colab or Kaggle)

**Before you start:** Colab → *Runtime → Change runtime type → T4 GPU*. Kaggle → *Settings → Accelerator → GPU T4 x2 (or P100)* and *Internet → On*.

Each team member runs the runs listed under their name in `lskd/experiments.py → ASSIGNMENT`. Every run saves a checkpoint after **every epoch** and can be resumed with `--resume`, so a disconnect costs at most one epoch — just re-run the training cell.

In [ ]:
!nvidia-smi

## 1. Get the code
Either clone your GitHub repo, or upload `lskd_project.zip` to the session and unzip it.

In [ ]:
import os
REPO_URL = ""   # e.g. "https://github.com/<your-user>/lskd.git"  (leave empty if you uploaded the zip)
if REPO_URL:
    !git clone -q {REPO_URL} lskd
elif os.path.exists("lskd_project.zip"):
    !unzip -q -o lskd_project.zip -d .
%cd lskd
!ls

## 2. Persistent storage (Colab only)
Save `runs/` to Google Drive so nothing is lost if the session dies. On Kaggle skip this cell — `/kaggle/working` is kept when you *Save Version*.

In [ ]:
IN_COLAB = "google.colab" in str(get_ipython())
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/lskd_runs", exist_ok=True)
    if not os.path.islink("runs"):
        !rm -rf runs && ln -s /content/drive/MyDrive/lskd_runs runs
!ls -la

## 3. Teacher weights, unit tests, smoke test

In [ ]:
!python scripts/download_teacher.py
!python -m pytest -q tests || python tests/test_core.py
!python train.py --preset c100_kd_ls --smoke

## 4. Train your assigned runs
Set `MEMBER`. Look at the epoch time printed in the log: ~10–20 s/epoch on a T4 means ~40–80 min per 240-epoch run.
If you are short of time, set `EPOCHS = 120` — but then **every** run in the comparison must use 120 (state it in the report).

In [ ]:
from lskd.experiments import ASSIGNMENT
MEMBER = "member1"     # member1 / member2 / member3
EPOCHS = 240
for preset in ASSIGNMENT[MEMBER]:
    print("=" * 30, preset)
    !python train.py --preset {preset} --epochs {EPOCHS} --resume
    if preset.endswith("teacher_resnet56"):
        pass  # c10 teacher is read directly from runs/c10_teacher_resnet56/best.pth by the c10 presets

## 5. Evaluate (run after the CIFAR-100 KD and KD+LS runs exist)

In [ ]:
!python evaluate.py --dataset cifar100 --teacher checkpoints/teachers/resnet56_cifar100.pth \
    --models "Vanilla KD=runs/c100_kd/final.pth" "KD+LS=runs/c100_kd_ls/final.pth"
# optional, if the CIFAR-10 runs exist:
# !python evaluate.py --dataset cifar10 --teacher runs/c10_teacher_resnet56/final.pth --models "Vanilla KD=runs/c10_kd/final.pth" "KD+LS=runs/c10_kd_ls/final.pth" 

## 6. Package results
Download `results_<member>.zip` (contains summaries, histories, final/best weights and eval files — the large `last.pth` files are left out) and send it to the team.

In [ ]:
!zip -q -r results_{MEMBER}.zip runs results -x "*/last.pth" "*_smoke/*"
!ls -lh results_{MEMBER}.zip